# 07. MAF で構築したエージェントを Hosted Agent としてデプロイする

## コードの解説

**Azure AI Foundry の Hosted Agent** は、エージェントを Azure 上に永続デプロイする機能です。
`azure-ai-projects` SDK を使ってエージェントの定義を Foundry に登録し、
API 経由でどこからでも呼び出せるサービスとして公開できます。

### Hosted Agent の仕組み

```
ローカル開発環境                    Azure AI Foundry
┌──────────────────────┐           ┌─────────────────────────┐
│  azure-ai-projects SDK│ ─deploy─→ │  Hosted Agent           │
│  (エージェント定義)   │           │  ┌─────────────────────┐ │
│                      │           │  │  instructions       │ │
│                      │ ─call──→  │  │  tools              │ │
│                      │           │  │  セッション管理      │ │
│                      │ ←result─  │  └─────────────────────┘ │
└──────────────────────┘           └─────────────────────────┘
```

### ローカルエージェントとの違い

| 項目 | ローカル MAF エージェント | Hosted Agent |
|-----|----------------------|-------------|
| 実行場所 | ローカル Python プロセス | Azure クラウド |
| 状態管理 | アプリ側で管理 | Foundry が自動管理 |
| スケーラビリティ | 手動 | 自動スケール |
| 呼び出し方法 | Python コード直接 | REST API / SDK |

## Azure の操作手順

### 1. Azure AI Foundry プロジェクトの作成

1. [Azure AI Foundry ポータル](https://ai.azure.com) にアクセス
2. 「新しいプロジェクト」をクリック
3. ハブを選択または新規作成（リージョン: eastus2 / swedencentral 推奨）
4. プロジェクト名を入力して作成

### 2. モデルのデプロイ

1. プロジェクト内の「モデル + エンドポイント」→「モデルのデプロイ」をクリック
2. `gpt-4o` または `gpt-4o-mini` を選択してデプロイ
3. デプロイ名をメモ（例: `gpt-4o`）

### 3. プロジェクトエンドポイントの確認

1. プロジェクトの「概要」ページを開く
2. **「Azure AI Foundry API」** セクションのエンドポイント URL をコピー
   - 形式: `https://<hub-name>.services.ai.azure.com/api/projects/<project-name>`

### 4. .env ファイルへの設定追加

```env
AZURE_AI_PROJECT_ENDPOINT=https://<hub-name>.services.ai.azure.com/api/projects/<project-name>
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4o
```

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MessageRole

load_dotenv()

AZURE_AI_PROJECT_ENDPOINT      = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_AI_MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")


# ---------------------------------------------------------------
# Azure AI Foundry に Hosted Agent をデプロイする
#
# AIProjectClient: Azure AI Foundry プロジェクトへの接続クライアント
# agents.create_agent(): エージェント定義を Foundry に登録（デプロイ）
# agents.create_thread(): 会話スレッドを作成（セッションに相当）
# agents.create_and_process_run(): エージェントを実行して結果を待機
# ---------------------------------------------------------------
with AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
) as project_client:

    # Hosted Agent の作成（Foundry 上にエージェント定義を登録）
    agent = project_client.agents.create_agent(
        model=AZURE_AI_MODEL_DEPLOYMENT_NAME,
        name="HostedTravelAssistant",
        instructions=(
            "あなたは Azure AI Foundry でホストされた旅行アシスタントです。"
            "旅行先のおすすめ情報や旅のコツをわかりやすく提供してください。"
        ),
    )
    print(f"Hosted Agent 作成完了: ID={agent.id}")

    # 会話スレッドの作成（マルチターン会話のセッション）
    thread = project_client.agents.threads.create()
    print(f"スレッド作成完了: ID={thread.id}")

    # ユーザーメッセージをスレッドに追加
    project_client.agents.messages.create(
        thread_id=thread.id,
        role=MessageRole.USER,
        content="東京旅行の見どころを 3 つ教えてください。",
    )

    # エージェントを実行して応答を待機
    run = project_client.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent.id,
    )
    print(f"実行ステータス: {run.status}")

    # 応答メッセージの取得
    messages = project_client.agents.messages.list(thread_id=thread.id)
    for msg in messages:
        if msg.role == MessageRole.AGENT:
            for content in msg.content:
                if hasattr(content, "text"):
                    print(f"\nエージェント応答:\n{content.text.value}")
                    break
            break

    # 後片付け（デプロイしたエージェントを削除）
    project_client.agents.delete_agent(agent.id)
    print("\nエージェント削除完了")